# 릿지 회귀
일반 선형 회귀에서는 변수들이 강하게 엮여있으면 계수가 불안정해질 수 있습니다. 릿지는 여기에서 예측 오차는 줄이되, 계수들이 너무 커지지 않게 하는 방식입니다.

# 손실(Loss)이란?

손실은 모델이 예측을 얼마나 틀렸는지 숫자로 나타내는 값을 말합니다.

대표적으로 `MSE = (실제값 - 예측값)^2 의 평균`과 같이 모델이 얼마나 못맞추었는지를 나타내는 것이 손실을 말합니다.

# 데이터들이 강하게 얽혀있다는 것은?
데이터가 비슷한 설명을 한다는 것은 요소들 중 비슷한 설명을 하는 (비슷한 상관성을 가진) 데이터들을 말합니다. 그중 하나로 상관계수에서 강한 양의 상관성을 띄는 데이터들이 강하게 얽혀있다고 볼 수 있습니다.

이렇게 강한 상관성을 가진 데이터들은 예측을 할 때, 모델이 어떤 것을 기준으로 계산할지 정확하지 못한 상태가 될 수 있습니다. 

예를 들어 `[A, B, C]` 속성이 모두 강한 양의 상관관계를 가진다면 회귀식에서 셋중 하나에 큰 가중치를 준다던가 골고루 영향을 주거나 하나에 음의 상관관계를 주고 나머지에 양의 계수를 줄 수도 있는 상황이 될 수 있습니다. 

여기에서 **릿지 회귀**는 이런 속성들의 계수들이 너무 커지지 않게 눌러주는 효과를 가져다줄 수 있습니다.

추가로 **랏쏘 회귀**는 덜중요한 변수 계수를 `0`으로 만들어 변수 선택 효과를 주기도 합니다.

# 릿지 회귀의 한계
릿지 회귀는 결국에 정답에 가까운 계수, 정확한 관계를 찾는 모델보다는 최악의, 이상한 결과를 막아주는 일종의 장치에 불과합니다. 실제로 눌러버린 계수가 진짜 관계인 경우가 있을 수 있습니다.

머신러닝에서 이것을 일반적으로 `bias-variance tradeoff`라고 합니다. (계수/기울기를 줄이는 대신 편향/절편을 높이는 방향)

In [8]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

data = fetch_ucirepo(id=9)

X = pd.DataFrame(data.data.features)
y = pd.DataFrame(data.data.targets)

In [3]:
X.head()

,displacement,cylinders,horsepower,weight,acceleration,model_year,origin
0,307.0,8,130.0,3504,12.0,70,1
1,350.0,8,165.0,3693,11.5,70,1
2,318.0,8,150.0,3436,11.0,70,1
3,304.0,8,150.0,3433,12.0,70,1
4,302.0,8,140.0,3449,10.5,70,1


In [10]:
y.head()

,mpg
0,18.0
1,15.0
2,18.0
3,16.0
4,17.0


In [11]:
df = pd.concat((X, y), axis=1)
df.head()

,displacement,cylinders,horsepower,weight,acceleration,model_year,origin,mpg
0,307.0,8,130.0,3504,12.0,70,1,18.0
1,350.0,8,165.0,3693,11.5,70,1,15.0
2,318.0,8,150.0,3436,11.0,70,1,18.0
3,304.0,8,150.0,3433,12.0,70,1,16.0
4,302.0,8,140.0,3449,10.5,70,1,17.0


In [16]:
print(df.isna().sum())
df = df.dropna(axis=0)

displacement    0
cylinders       0
horsepower      6
weight          0
acceleration    0
model_year      0
origin          0
mpg             0
dtype: int64


In [26]:
from sklearn.model_selection import train_test_split

X = df[df.drop(columns=["mpg"]).columns]
y = df["mpg"]

X_train, X_test, y_train, y_test = train_test_split(
  X, y,
  test_size=0.2,
  random_state=42
)

In [27]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocess = ColumnTransformer(
  transformers=[
    ("num", StandardScaler(), X.drop(columns=['origin']).columns),
    ("cat", OneHotEncoder(drop="first"), ["origin"])
  ]
)

In [29]:
import numpy as np

X_train_preprocessed = preprocess.fit_transform(X_train)

X_train_preprocessed.shape
# np.where(np.isnan(X_train_preprocessed))

(313, 8)

In [33]:
from sklearn.linear_model import Ridge
from sklearn import set_config

set_config(display="text")

model = Ridge(alpha=1.0) # alpha는 계수를 누르는 정도 (기본값이 1)

model.fit(X_train_preprocessed, y_train)


Ridge()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_test_preprocessed = preprocess.transform(X_test)

y_pred = model.predict(X_test_preprocessed)

# print(y_pred)
# print(y_test)

# print("랏쏘 회귀 R²:", model.score(X_test_preprocessed, y_test))
print("릿지 회귀 R2: ", r2_score(y_test, y_pred))
print("릿지 회귀 MAE: ", mean_absolute_error(y_test, y_pred))
print("릿지 회귀 RMSE: ", np.sqrt(mean_squared_error(y_test, y_pred)))

랏쏘 회귀 R2:  0.8104121264748629
랏쏘 회귀 MAE:  2.456596505018637
랏쏘 회귀 RMSE:  3.257306515093934


In [ ]:
from sklearn.linear_model import LinearRegression

model_linear_regression = LinearRegression()

model_linear_regression.fit(X_train_preprocessed, y_train)

y_pred = model_linear_regression.predict(X_test_preprocessed)

# print(y_pred)
# print(y_test)

# print("선형 회귀 R²:", model_linear_regression.score(X_test_preprocessed, y_test))
print("선형 회귀 R2: ", r2_score(y_test, y_pred))
print("선형 회귀 MAE: ", mean_absolute_error(y_test, y_pred))
print("선형 회귀 RMSE: ", np.sqrt(mean_squared_error(y_test, y_pred)))

선형 회귀 R2:  0.8122496423097463
선형 회귀 MAE:  2.4619996980661494
선형 회귀 RMSE:  3.2561140968474023


In [ ]:
# 선형 모델과 릿지 모델의 계수를 비교해보자
line_result = np.append(model_linear_regression.coef_, model_linear_regression.intercept_)
ridge_result = np.append(model.coef_, model.intercept_)
pd.DataFrame({
  "선형 모델": line_result,
  "릿지 모델": ridge_result,
  "선형 - 릿지": (line_result - ridge_result)
}, X.columns.drop('origin').tolist() + ['origin_2', 'origin_3', 'bias'])




# for lin, rid, col in zip(model_linear_regression.coef_, model.coef_, X.columns):


,선형 모델,랏쏘 모델,선형 - 랏쏘
displacement,1.989750,1.641771,0.347979
cylinders,-0.580795,-0.506213,-0.074582
horsepower,-0.826385,-0.828251,0.001866
weight,-5.393692,-5.196141,-0.197552
acceleration,0.118718,0.080674,0.038045
model_year,2.889178,2.855171,0.034007
origin_2,2.875499,2.660780,0.214720
origin_3,3.205969,3.041928,0.164041
bias,22.513099,22.579045,-0.065946


# 결과를 보면
결과를 보면 배기량이 변화 절대값이 제일 큰것을 확인할 수 있다. 실제로 계수는 무게가 가장 높은데도.

이는 배기량이 다른 요소들 (예를 들면 무게, 마력, 실린더 등)을 설명하기 때문에 변화 폭이 더 넓어지는 것이다.